# Fairness Evaluation on UCI Adult Dataset
This notebook:
- Loads and preprocesses the UCI Adult dataset
- Trains a classifier to predict income level
- Evaluates fairness using Fairlearn

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference, equalized_odds_difference
import json

## 1. Load and Preprocess the Data

In [8]:
# Load dataset
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]

df = pd.read_csv("adult/adult.data", header=None, names=column_names, na_values=" ?")
df = df.dropna().reset_index(drop=True)
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

## 2. Encode and Split Data

In [9]:

# Label encoding for categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.drop("income")
df_encoded = df.copy()
le = LabelEncoder()
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Target variable
df_encoded["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# Features and labels
X = df_encoded.drop(columns="income")
y = df_encoded["income"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
sensitive = df.loc[X_test.index, "sex"]
sensitive_cols = ['sex', 'race', 'education']


## 3. Train Classifier and Predict

In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [5]:
#train with diffent models
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

/home/gfragi/postDoc/haic_benchmark_suite/bench-env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
# Build a dict of the sensitive feature arrays:
sensitive_dict = {}
for col in sensitive_cols:
    sensitive_dict[col] = df.loc[X_test.index, col].tolist()



output = {
        "predictions": list(map(int, y_pred)),
        "labels":      list(map(int, y_test)),
        "sensitive_features": sensitive_dict
}

# Save the output to a JSON file
with open("fairness_output.json", "w") as f:
    json.dump(output, f, indent=4)

## 4. Fairness Evaluation

In [12]:
mf = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive
)

print("Accuracy by group:")
print(mf.by_group["accuracy"])

print("\nSelection rate by group:")
print(mf.by_group["selection_rate"])

print("\nDemographic parity difference:",
      demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive))

print("Equalized odds difference:",
      equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive))

Accuracy by group:
sex
Female    0.928547
Male      0.820137
Name: accuracy, dtype: float64

Selection rate by group:
sex
Female    0.088629
Male      0.272727
Name: selection_rate, dtype: float64

Demographic parity difference: 0.18409793572967736
Equalized odds difference: 0.07287480654303938
